In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [15]:
# LOAD DATA
DATA_DIR = "../data/processed/"
adasyn_data = pd.read_parquet(f"{DATA_DIR}adasyn_balanced.parquet")
smote_data = pd.read_parquet(f"{DATA_DIR}smote_balanced.parquet")
equal_undersampled_data = pd.read_parquet(f"{DATA_DIR}equal_undersampled.parquet")
moderate_undersampled_data = pd.read_parquet(f"{DATA_DIR}moderate_undersampled.parquet")
NearMiss_equal_data = pd.read_parquet(f"{DATA_DIR}NearMiss_equal.parquet")
NearMiss_moderate_data = pd.read_parquet(f"{DATA_DIR}NearMiss_moderate.parquet")
original_data = pd.read_parquet(f"{DATA_DIR}processed_data.parquet")
tomek_links_data = pd.read_parquet(f"{DATA_DIR}tomek_links.parquet")

In [16]:
selected_features = ['Elevation', 'Horizontal_Distance_To_Hydrology', 
                    'Horizontal_Distance_To_Roadways', 'Hillshade_Noon',
                    'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area1',
                    'Wilderness_Area3', 'Wilderness_Area4', 
                    'Soil_Type2', 'Soil_Type4', 'Soil_Type10', 'Soil_Type12',
                    'Soil_Type22', 'Soil_Type23', 'Soil_Type38', 'Soil_Type39',
                    'Euclidean_Distance_To_Hydrology', 'Distance_To_Hydrology_To_Roadways_Ratio',
                    'Total_Distance', 'Aspect_North_South', 'Cover_Type'] 



# Dataset Selection

In this section we will choose between the balanced datasets. We will train a RandomForest with same hyperparameters for each of the datasets and test the performance so that we can find the dataset with the best results.

In [23]:
def split_features_and_target(df, selected_features):
    df = df[selected_features]
    print(df.shape)
    X = df.drop('Cover_Type', axis=1)
    y = df['Cover_Type']
    return X,y

In [24]:
# PARAMETROS
RANDOM_STATE = 42
N_SPLITS = 5

hyperparameters = {
    'n_estimators': 100,
    'max_depth': 15,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'random_state': RANDOM_STATE
}

print(f"Hiperparámetros a usar: {hyperparameters}\n")

Hiperparámetros a usar: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2, 'random_state': 42}



In [25]:
datasets = {
    'SMOTE': smote_data,
    'ADASYN': adasyn_data,
    'Random_Undersampling_Moderate': moderate_undersampled_data,
    'Random_Undersampling_Equal': equal_undersampled_data,
    'NearMiss_Moderate': NearMiss_moderate_data,
    'NearMiss_Equal': NearMiss_equal_data,
    'Tomek_Links': tomek_links_data,
}

In [26]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'Accuracy': make_scorer(accuracy_score),
    'Precision': make_scorer(precision_score, average='weighted', zero_division=0),
    'Recall': make_scorer(recall_score, average='weighted', zero_division=0),
    'F1': make_scorer(f1_score, average='weighted', zero_division=0)
}

results = {}
for dataset_name, data in datasets.items():
    print(f"\n{'='*70}")
    print(f"Processing: {dataset_name}")
    print(f"{'='*70}")
    
    # Separar X, y
    X, y = split_features_and_target(data, selected_features=selected_features)
    
    # Crear modelo
    rf_model = RandomForestClassifier(**hyperparameters, n_jobs=-1)
    
    # Pipeline con scaler
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', rf_model)
    ])
    
    # Cross-Validation
    print(f"Executing {N_SPLITS}-Fold Cross Validation...")
    cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=True)
    
    # Calcular promedios
    test_acc = cv_results['test_Accuracy'].mean()
    test_acc_std = cv_results['test_Accuracy'].std()
    test_f1 = cv_results['test_F1'].mean()
    test_f1_std = cv_results['test_F1'].std()
    test_precision = cv_results['test_Precision'].mean()
    test_recall = cv_results['test_Recall'].mean()
    
    train_acc = cv_results['train_Accuracy'].mean()
    
    # Guardar resultados
    results[dataset_name] = {
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Test Accuracy Std': test_acc_std,
        'Test Precision': test_precision,
        'Test Recall': test_recall,
        'Test F1': test_f1,
        'Test F1 Std': test_f1_std,
        'Samples': len(X)
    }


Processing: SMOTE
(1982778, 21)
Executing 5-Fold Cross Validation...


KeyboardInterrupt: 

In [4]:
print(smote_data['Cover_Type'])

0          5
1          5
2          2
3          2
4          5
          ..
1982773    7
1982774    7
1982775    7
1982776    7
1982777    7
Name: Cover_Type, Length: 1982778, dtype: int64


In [ ]:
print("COMPARACIÓN DE DATASETS")
print(f"{'='*70}\n")

results_df = pd.DataFrame(results).T
print(results_df)

# Resumen
print(f"\n{'='*70}")
print("🏆 MEJORES DATASETS")
print(f"{'='*70}")

best_acc = results_df['Test Accuracy'].idxmax()
best_f1 = results_df['Test F1'].idxmax()

print(f"\n✅ Mejor Accuracy: {best_acc}")
print(f"   Accuracy: {results_df.loc[best_acc, 'Test Accuracy']:.4f}")
print(f"   F1-Score: {results_df.loc[best_acc, 'Test F1']:.4f}")

print(f"\n✅ Mejor F1-Score: {best_f1}")
print(f"   Accuracy: {results_df.loc[best_f1, 'Test Accuracy']:.4f}")
print(f"   F1-Score: {results_df.loc[best_f1, 'Test F1']:.4f}")

# Ordenar por Accuracy descendente
print(f"\n📋 Ranking por Test Accuracy:")
ranking = results_df[['Test Accuracy', 'Test F1']].sort_values('Test Accuracy', ascending=False)
for idx, (dataset, row) in enumerate(ranking.iterrows(), 1):
    print(f"  {idx}. {dataset:30s} - Accuracy: {row['Test Accuracy']:.4f}, F1: {row['Test F1']:.4f}")